# O primeiro cálculo da Arena Analytics no Spark

## Conhecendo o Spark por dentro

O cálculo de gols por Copa, agora no Spark, e o que acontece por dentro para ele rodar.

## Termos

- **Processo:** um programa em execução.
- **Núcleo (core):** a parte do processador que executa uma tarefa por vez.
- **Nó:** cada computador de um cluster.

## As peças de uma aplicação Spark

| Peça | O que é |
|---|---|
| **Driver** | o processo onde o seu programa roda (o notebook). Cria a `SparkSession`, monta o plano de execução, divide em tarefas, distribui e recebe o resultado. Coordena, mas não processa o volume pesado |
| **Cluster manager** | o sistema que controla as máquinas do cluster e empresta núcleos e memória (YARN, Kubernetes, standalone) |
| **Worker** | cada máquina (nó) do cluster que oferece núcleos e memória |
| **Executor** | o processo, dentro de um worker, que lê o dado, calcula e guarda resultados intermediários |
| **Partição** | cada pedaço em que o dado foi dividido |
| **Task** | o trabalho de aplicar o plano a **uma** partição. Uma partição gera uma task |

```
                         seu notebook
                   +----------------------+
                   |        DRIVER        |  monta o plano e distribui as tasks
                   +----------------------+
                              |
                   +----------------------+
                   |    CLUSTER MANAGER   |  empresta máquinas (YARN, Kubernetes...)
                   +----------------------+
                     |                  |
       +-------------------+     +-------------------+
       |  WORKER 1 (nó)    |     |  WORKER 2 (nó)    |   máquinas do cluster
       |  +-------------+  |     |  +-------------+  |
       |  |  EXECUTOR   |  |     |  |  EXECUTOR   |  |   processos que fazem o trabalho
       |  | task | task |  |     |  | task | task |  |   uma task por núcleo, por vez
       |  +-------------+  |     |  +-------------+  |
       +-------------------+     +-------------------+
```

## O cálculo de gols por Copa num cluster

1. O notebook descreve o cálculo: ler as partidas, calcular os gols, agrupar por ano e somar.
2. O **driver** transforma isso num plano e divide o arquivo em partições.
3. O driver pede recursos ao **cluster manager**, que sobe **executors** nos **workers**.
4. O driver envia **uma task para cada partição**.
5. Cada **executor** soma os gols do seu pedaço, por Copa (somas parciais).
6. Os executors **trocam dados entre si** para os parciais de cada Copa se encontrarem: o **shuffle**.
7. Cada Copa recebe a sua soma final.
8. O resultado volta para o **driver** e aparece no notebook.

## Modo local

Não existe cluster manager: o driver e o executor ficam **no mesmo processo**, dentro do container, e cada núcleo
executa uma task.

```
              container (sua máquina)
   +-----------------------------------------------+
   |   um único processo Spark                      |
   |   DRIVER + EXECUTOR                            |
   |   núcleo 1: task   núcleo 2: task   ...        |
   +-----------------------------------------------+
```

O código é o mesmo de um cluster; muda o `master` da sessão.

## A SparkSession

| Parâmetro | O que faz |
|---|---|
| `.appName("arena-gols-por-copa")` | o nome da aplicação; aparece na Spark UI e nos logs. Não muda o comportamento |
| `.master("local[*]")` | onde roda e com quantos núcleos: `local[*]` usa todos, `local[2]` usaria 2. Num cluster, o endereço dele (`yarn`, `spark://host:porta`) |
| `.config("spark.sql.warehouse.dir", ...)` | a pasta das tabelas que o Spark gerencia (aula 3) |
| `.getOrCreate()` | cria a sessão, ou devolve a que já está rodando |

A última linha, `spark`, mostra um resumo da sessão com a versão e o link da Spark UI.

`defaultParallelism`: quantas tasks o Spark executa **ao mesmo tempo** nesta máquina.

## A Spark UI

Página de acompanhamento da sessão, no navegador. Cada notebook tem a sua sessão e a sua porta: a primeira usa a
**4040**, a seguinte a 4041, e assim por diante. Endereço desta sessão:

O Spark mostra o nome interno do container; no navegador, use `localhost`.

| Aba | O que mostra |
|---|---|
| **Jobs** | cada ação executada, com o tempo de cada uma |
| **Stages** | as etapas de cada job e as tarefas de cada etapa |
| **Storage** | os dados guardados em cache |
| **Environment** | todas as configurações da sessão |
| **Executors** | os processos que executam as tarefas, com memória e disco |
| **SQL / DataFrame** | o plano de cada consulta, desenhado |

## Partições

Ao ler arquivos, o Spark divide em pedaços de até **128 MB** (padrão). O arquivo de partidas tem 68 KB: uma
partição. Abaixo, o limite é reduzido para 16 KB só para visualizar a divisão; não se faz isso em projeto.

Cada partição gera uma task, e as tasks rodam em paralelo, até o número de núcleos disponíveis.

## Gols por Copa com Spark

Os mesmos passos do pandas: filtrar, calcular, agrupar e somar. As funções são detalhadas na aula 2.

## Transformações e ações

- **Transformações** (`where`, `withColumn`, `groupBy(...).agg(...)`, `select`, `join`...): só **descrevem** um
  passo do plano. Nada é executado.
- **Ações** (`show`, `count`, `collect`, `write`...): pedem um resultado. Só então o driver envia as tasks para os
  executors.

Esse comportamento é a **lazy evaluation** (avaliação preguiçosa): como o Spark enxerga o plano inteiro antes de
executar, ele pode otimizá-lo.

```
seu código  ->  plano lógico  ->  Catalyst otimiza  ->  plano físico  ->  tasks
               (o que fazer)                          (como fazer)
```

O **Catalyst** é o otimizador do Spark: reescreve o plano antes de executar, por exemplo juntando filtros ou
deixando de ler colunas que o cálculo não usa.

## O plano de execução

`explain()` mostra o plano montado pelo driver. Ele se lê **de baixo para cima**.

| Linha do plano | O que significa |
|---|---|
| `FileScan csv` | cada task lê a sua partição do arquivo (só as colunas necessárias) |
| `Filter` | pula as linhas sem ano |
| `Project` | calcula os gols de cada partida |
| `HashAggregate ... partial_sum` | **soma parcial**: cada task soma os gols do seu pedaço, por Copa |
| `Exchange hashpartitioning(Year...)` | **shuffle**: a troca de dados para os parciais de cada Copa se encontrarem |
| `HashAggregate ... sum` | **soma final** de cada Copa |

`partial_sum` e `Exchange` são as duas fases do processamento distribuído: calcular em paralelo e juntar.

## As tasks na Spark UI

Na aba **Stages**, o stage com **5 tasks** é o processamento das 5 partições. Stages com 1 task ou marcados como
"skipped" são etapas que o Spark reaproveitou ou juntou (aula 4).

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()